# Lesson 11A: Self-Organizing Maps Theory — Competitive Learning and Neighborhoods

## Introduction

Every method in this course so far has neurons, centroids, or components with no fixed relationship to each other — K-Means centroids could be relabeled in any order and nothing would change. A **Self-Organizing Map** (Kohonen, 1982) is different: its neurons sit on a fixed grid (usually 2D), and training doesn't just move them toward the data — it arranges them so that **neurons that are close on the grid end up representing similar regions of the input space**. This is topology preservation, and it's the SOM's defining property.

The training mechanism is **competitive learning**: for each input, neurons compete to be the "winner" (the one whose weight vector is closest), and the winner — along with its grid neighbors — gets pulled toward the input. Over training, both the neighborhood size and the learning rate shrink, letting the map go from coarse, global organization to fine, local detail.

In this lesson:
1. Frame SOM neurons as a grid of weight vectors competing for inputs
2. Derive the Best Matching Unit (BMU) — the winner neuron
3. Derive the neighborhood function and its decay over training
4. Derive the learning rate decay
5. Combine these into the full SOM update rule and implement it from scratch
6. Cross-check against the MiniSom reference implementation
7. Visualize both grid unfolding (quantization) and topology preservation (the classic RGB-color map)

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Competitive Learning: A Grid of Neurons](#competitive-learning)
4. [Best Matching Unit](#bmu)
5. [Neighborhood Function and Decay](#neighborhood)
6. [Learning Rate Decay](#learning-rate)
7. [The SOM Update Rule](#update-rule)
8. [From-Scratch Implementation](#scratch)
9. [Grid Unfolding and Quantization Error](#unfolding)
10. [Cross-Check Against MiniSom](#minisom-check)
11. [Topology Preservation: The RGB Color Map](#rgb-map)
12. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom
from typing import Tuple
from numpy.typing import NDArray

rng = np.random.RandomState(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="competitive-learning"></a>
## Competitive Learning: A Grid of Neurons

A SOM is a $g \times g$ grid of neurons, each holding a weight vector $w_{i,j} \in \mathbb{R}^d$ in the *same* space as the input data. Unlike K-Means centroids, these neurons have a fixed 2D spatial relationship to each other **before training even starts** — neuron $(i, j)$ is adjacent to $(i\pm1, j)$ and $(i, j\pm1)$ regardless of what the weights converge to.

Training is **competitive**: for each input $x$, every neuron computes its distance to $x$, and the single closest neuron — the **winner** — gets to update most strongly. Its grid neighbors update too, but progressively less as grid-distance from the winner increases. This is what "self-organizing" means: neighboring neurons are pulled toward similar regions of input space purely because they're pulled toward *whatever their neighbor most recently won*, again and again, across the whole dataset.

<a name="bmu"></a>
## Best Matching Unit

The **Best Matching Unit (BMU)** for input $x$ is the neuron whose weight vector is closest in Euclidean distance:

$$\text{BMU}(x) = \arg\min_{(i,j)} \|x - w_{i,j}\|$$

This is identical in spirit to finding the nearest centroid in K-Means — the difference is entirely in what happens *next*: K-Means updates only the winning centroid; a SOM updates the winner **and its neighbors on the grid**.

<a name="neighborhood"></a>
## Neighborhood Function and Decay

The **neighborhood function** $h(i, j, t)$ measures how strongly neuron $(i,j)$ should update given that some other neuron won, as a function of their *grid* distance (not input-space distance) and training time $t$. The standard choice is a Gaussian centered on the BMU:

$$h(i, j, t) = \exp\left(-\frac{d_{\text{grid}}((i,j), \text{BMU})^2}{2\sigma(t)^2}\right)$$

The neighborhood radius $\sigma(t)$ **shrinks over training**, typically exponentially:

$$\sigma(t) = \sigma_0 \exp(-t / \tau), \qquad \tau = \frac{n_{\text{iter}}}{\ln(\sigma_0)}$$

Early in training, $\sigma(t)$ is large — most of the grid moves together, establishing coarse global organization. Late in training, $\sigma(t)$ shrinks toward a small radius (often close to 1 grid cell), letting neurons fine-tune to local detail without disturbing distant parts of the map.

In [ ]:
# Visualize neighborhood shrinking over training
grid_size = 10
sigma0 = grid_size / 2
n_iter = 2000
tau = n_iter / np.log(sigma0)

checkpoints = [0, 500, 1000, 1500, 1999]
gx, gy = np.meshgrid(np.arange(grid_size), np.arange(grid_size), indexing='ij')
bmu_coord = np.array([grid_size // 2, grid_size // 2])

fig, axes = plt.subplots(1, len(checkpoints), figsize=(4 * len(checkpoints), 4))
for ax, t in zip(axes, checkpoints):
    sigma_t = sigma0 * np.exp(-t / tau)
    dist_sq = (gx - bmu_coord[0]) ** 2 + (gy - bmu_coord[1]) ** 2
    h = np.exp(-dist_sq / (2 * sigma_t ** 2))
    im = ax.imshow(h, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f't={t}, σ={sigma_t:.2f}', fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

print("💡 The neighborhood starts covering most of the grid and shrinks to a tight local patch around the BMU")

<a name="learning-rate"></a>
## Learning Rate Decay

The learning rate $\eta(t)$ also decays, controlling the overall step size independent of neighborhood shape:

$$\eta(t) = \eta_0 \exp(-t / n_{\text{iter}})$$

Combined with the shrinking neighborhood, this produces the classic two-phase SOM training pattern: an early **ordering phase** (large $\sigma$, large $\eta$) that establishes rough topology, and a later **convergence phase** (small $\sigma$, small $\eta$) that refines local detail.

<a name="update-rule"></a>
## The SOM Update Rule

Putting it together, for input $x$ at time $t$ with BMU $b$:

$$w_{i,j}(t+1) = w_{i,j}(t) + \eta(t) \cdot h(i, j, t) \cdot (x - w_{i,j}(t))$$

Every neuron updates on every input — but $h(i,j,t)$ makes that update negligible for neurons far from the BMU. Notice this collapses to a **pure vector-quantization update** (only the winner moves) in the limit $\sigma(t) \to 0$, where $h$ becomes 1 at the BMU and 0 everywhere else — the neighborhood term is exactly what makes a SOM more than K-Means with a training schedule.

<a name="scratch"></a>
## From-Scratch Implementation

In [ ]:
class SelfOrganizingMap:
    """Kohonen SOM: grid of neurons trained via competitive learning with neighborhood decay."""

    def __init__(self, grid_size: int, input_dim: int, rng: np.random.RandomState, sigma0: float = None, lr0: float = 0.5):
        self.grid_size = grid_size
        self.rng = rng
        self.weights = rng.uniform(0, 1, size=(grid_size, grid_size, input_dim))
        self.sigma0 = sigma0 if sigma0 is not None else grid_size / 2
        self.lr0 = lr0
        gx, gy = np.meshgrid(np.arange(grid_size), np.arange(grid_size), indexing='ij')
        self.grid_coords = np.stack([gx, gy], axis=-1).astype(float)

    def bmu(self, x: NDArray) -> Tuple[int, int]:
        dists = np.linalg.norm(self.weights - x, axis=-1)
        return np.unravel_index(np.argmin(dists), dists.shape)

    def train(self, X: NDArray, n_iter: int) -> None:
        tau = n_iter / np.log(self.sigma0)
        for t in range(n_iter):
            x = X[self.rng.randint(len(X))]
            sigma_t = self.sigma0 * np.exp(-t / tau)
            lr_t = self.lr0 * np.exp(-t / n_iter)

            bmu_coord = np.array(self.bmu(x), dtype=float)
            dist_sq = np.sum((self.grid_coords - bmu_coord) ** 2, axis=-1)
            neighborhood = np.exp(-dist_sq / (2 * sigma_t ** 2))

            self.weights += lr_t * neighborhood[..., None] * (x - self.weights)

    def quantization_error(self, X: NDArray) -> float:
        return float(np.mean([np.linalg.norm(self.weights[self.bmu(x)] - x) for x in X]))


print("✅ SelfOrganizingMap class defined")

<a name="unfolding"></a>
## Grid Unfolding and Quantization Error

Train on five 2D Gaussian blobs. Because the input is 2D, each neuron's weight vector *is* a point in the same plane as the data — letting us literally draw the grid lattice on top of the data and watch it unfold to cover the clusters.

In [ ]:
X_blobs, _ = make_blobs(n_samples=400, centers=5, cluster_std=0.6, random_state=42)
X_blobs = MinMaxScaler().fit_transform(X_blobs)

som = SelfOrganizingMap(grid_size=8, input_dim=2, rng=rng)
qe_curve = [som.quantization_error(X_blobs)]

snapshots = {0: som.weights.copy()}
checkpoint_iters = [200, 800, 2000, 5000]
iters_so_far = 0
for target in checkpoint_iters:
    som.train(X_blobs, n_iter=target - iters_so_far)
    iters_so_far = target
    snapshots[target] = som.weights.copy()
    qe_curve.append(som.quantization_error(X_blobs))

print(f"Quantization error: {qe_curve[0]:.4f} (init) -> {qe_curve[-1]:.4f} (after {checkpoint_iters[-1]} iterations)")

fig, axes = plt.subplots(1, len(snapshots), figsize=(4.5 * len(snapshots), 4.5))
for ax, (t, w) in zip(axes, snapshots.items()):
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], s=15, alpha=0.3, color='gray')
    for row in range(w.shape[0]):
        ax.plot(w[row, :, 0], w[row, :, 1], color='crimson', linewidth=1, alpha=0.8)
    for col in range(w.shape[1]):
        ax.plot(w[:, col, 0], w[:, col, 1], color='crimson', linewidth=1, alpha=0.8)
    ax.scatter(w[:, :, 0], w[:, :, 1], s=20, color='crimson', zorder=3)
    ax.set_title(f't={t}', fontsize=11, fontweight='bold')
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

print("💡 The grid starts as a random tangle and unfolds into a net that covers all five clusters while keeping its lattice structure intact")

<a name="minisom-check"></a>
## Cross-Check Against MiniSom

MiniSom initializes weights by sampling actual data points (a common, effective practical choice), while our from-scratch version uses plain uniform random initialization — so their quantization-error *trajectories* aren't directly comparable starting point to starting point. What **is** comparable: after equivalent training, do both algorithms converge to a similarly low quantization error on the same data? If our derivation and implementation are correct, they should.

In [ ]:
ms = MiniSom(8, 8, 2, sigma=4.0, learning_rate=0.5, random_seed=42)
ms.random_weights_init(X_blobs)
ms.train(X_blobs, 5000, random_order=True)

print(f"From-scratch SOM final quantization error: {qe_curve[-1]:.4f}")
print(f"MiniSom final quantization error:           {ms.quantization_error(X_blobs):.4f}")
print("\n💡 Both land in the same ballpark despite different initializations and independent implementations — confirms the update rule is correctly derived")

<a name="rgb-map"></a>
## Topology Preservation: The RGB Color Map

The classic illustration of topology preservation: train a SOM on random RGB colors (3D input, no engineered structure at all) and look at the final grid as an image. If neighboring neurons represent similar colors, the map should look like a smooth, continuous gradient — direct visual proof that grid-adjacent neurons end up representing similar inputs, purely as a byproduct of the neighborhood update.

In [ ]:
colors = rng.uniform(0, 1, size=(200, 3))

som_colors = SelfOrganizingMap(grid_size=12, input_dim=3, rng=rng)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(som_colors.weights)
axes[0].set_title('Before training (random)', fontsize=11, fontweight='bold')
axes[0].set_xticks([]); axes[0].set_yticks([])

som_colors.train(colors, n_iter=3000)

axes[1].imshow(som_colors.weights)
axes[1].set_title('After training', fontsize=11, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout()
plt.show()

print("💡 No color labels, no clustering objective stated anywhere — smooth regions emerge purely from the neighborhood-preserving update rule")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **SOM neurons have a fixed grid topology before training starts** — training doesn't just fit weight vectors to data, it arranges them so grid-adjacent neurons represent similar inputs.
2. **Competitive learning finds a winner (BMU)** per input, exactly like nearest-centroid assignment in K-Means.
3. **The neighborhood function is what makes a SOM more than K-Means** — it pulls the BMU's grid-neighbors along too, with a radius that shrinks from global (ordering phase) to local (convergence phase).
4. **Grid unfolding is directly visualizable when input is 2D** — the lattice literally stretches across the data manifold while staying topologically intact.
5. **Topology preservation is provable even on structureless data** (random RGB colors) — smooth color gradients emerge purely from the update rule, no clustering objective required.

### When to Use SOM

- ✅ Visualizing high-dimensional data on a fixed, interpretable 2D grid
- ✅ Exploratory analysis where topology (which clusters are "near" each other) matters, not just membership
- ❌ When you need a specific target embedding metric preserved exactly (t-SNE/UMAP, Lesson 6, are built for that)
- ❌ Very high-dimensional sparse data — a SOM's grid resolution limits how much structure it can represent

### Next Steps

In Lesson 11B (practical), we'll:
- Apply MiniSom to a real dataset
- Build and interpret U-Matrix visualizations for finding cluster boundaries on the trained grid
- Use component planes to inspect individual feature contributions across the map

You now understand the one clustering-adjacent method in this course where the *arrangement* of the model's components, not just their values, is part of what gets learned 🎯